# 📊 Shopping Analytics Dashboard
Connects to `data/events.db` to compute daily KPIs.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

# Connect to DB
conn = sqlite3.connect('../data/events.db')

# Load Events
df_events = pd.read_sql_query("SELECT * FROM events", conn)
df_tracks = pd.read_sql_query("SELECT * FROM tracks_summary", conn)

## 1. Footfall & Conversion

In [ ]:
# Total Visitors (Count of unique tracks that started)
total_visitors = df_tracks['track_id'].nunique()
print(f"Total Visitors Tracked: {total_visitors}")

# Conversion Proxy: Visited Checkout vs Total
visited_checkout = df_tracks[df_tracks['has_checkout'] == 1]['track_id'].nunique()
conversion_rate = (visited_checkout / total_visitors * 100) if total_visitors > 0 else 0
print(f"Visited Checkout: {visited_checkout} ({conversion_rate:.2f}%)")

# No Purchase Exits
no_purchase_count = len(df_events[df_events['event_type'] == 'no_purchase_exit'])
print(f"No-Purchase Exits Flagged: {no_purchase_count}")

## 2. Dwell Time Analysis

In [ ]:
# Filter for dwell_end events to get durations
dwell_events = df_events[df_events['event_type'] == 'dwell_end']

avg_dwell_by_zone = dwell_events.groupby('zone_name')['duration_s'].mean()
print("\nAverage Dwell Time per Zone:")
print(avg_dwell_by_zone)

# Plot
avg_dwell_by_zone.plot(kind='bar', title='Avg Dwell Time (s) by Zone', figsize=(10, 6))
plt.ylabel('Seconds')
plt.show()

## 3. Worker Analytics

In [ ]:
# Identify workers
workers = df_tracks[df_tracks['role'] == 'worker']
print(f"Identified Workers: {workers['track_id'].nunique()}")

# Total Worker Hours in Cabin (Sum of all cabin dwell times for workers)
worker_cabin_time = dwell_events[
    (dwell_events['zone_name'] == 'worker_cabin') & 
    (dwell_events['track_id'].isin(workers['track_id']))
]['duration_s'].sum()

print(f"Total Worker Cabin Hours: {worker_cabin_time / 3600:.2f} hours")